# ⚡ Gleichzeitige Agenten-Workflows mit Microsoft Foundry (Python)

## 📋 Fortgeschrittenes Tutorial zur parallelen Verarbeitung

Dieses Notebook demonstriert **gleichzeitige Workflow-Muster** unter Verwendung des Microsoft Agent Frameworks. Sie lernen, wie man leistungsstarke, parallele Workflows baut, bei denen mehrere KI-Agenten gleichzeitig ausgeführt werden, was den Durchsatz erheblich verbessert und komplexe mehrfädige Geschäftsprozesse ermöglicht.

> **Migrationshinweis:** Dieses Beispiel verwies zuvor auf GitHub Models. GitHub Models wird eingestellt (Rückzug Juli 2026), daher verwendet es jetzt **Microsoft Foundry** über den `FoundryChatClient`, der auf die Azure OpenAI **Responses API** zugreift.

## 🎯 Lernziele

### 🚀 **Grundlagen paralleler Verarbeitung**
- **Parallele Agentenausführung**: Mehrere Agenten gleichzeitig für maximale Effizienz ausführen
- **Workflow-Orchestrierung**: Gleichzeitige Operationen koordinieren und gleichzeitig Datenkonsistenz wahren
- **Leistungsoptimierung**: Signifikante Beschleunigung durch parallele Verarbeitung erreichen
- **Ressourcenmanagement**: KI-Modellressourcen effizient über gleichzeitig laufende Operationen nutzen

### 🏗️ **Fortgeschrittene Nebenläufigkeitsmuster**
- **Fork-Join-Verarbeitung**: Arbeit auf mehrere Agenten aufteilen und Ergebnisse zusammenführen
- **Pipeline-Parallelität**: Überlappende Ausführungsphasen für kontinuierlichen Durchsatz
- **Lastverteilung**: Arbeit gleichmäßig auf verfügbare Agentenressourcen verteilen
- **Synchronisationspunkte**: Gleichzeitige Agenten an kritischen Workflow-Phasen koordinieren

### 🏢 **Parallele Unternehmensanwendungen**
- **Dokumentenverarbeitung mit hohem Volumen**: Mehrere Dokumente gleichzeitig verarbeiten
- **Echtzeit-Inhaltsanalyse**: Gleichzeitige Analyse eingehender Datenströme
- **Optimierung der Batch-Verarbeitung**: Durchsatz für groß angelegte Vorgänge maximieren
- **Multimodale Analyse**: Parallele Verarbeitung unterschiedlicher Inhaltstypen (Text, Bilder, Daten)

## ⚙️ Voraussetzungen & Einrichtung

### 📦 **Erforderliche Abhängigkeiten**

Installieren Sie das Agent Framework mit gleichzeitigen Workflow-Funktionalitäten:

```bash
pip install agent-framework -U
```

### 🔑 **Microsoft Foundry Konfiguration**

Melden Sie sich mit der Azure CLI (`az login`) an, damit sich `AzureCliCredential` authentifizieren kann, und legen Sie dann Ihre Microsoft Foundry-Projektdetails fest.

**Umgebungskonfiguration (.env-Datei):**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-5-mini
```

**Aspekte der parallelen Verarbeitung:**
- **Rate Limits**: Überwachen Sie Azure OpenAI-Rate-Limits für gleichzeitige Anfragen
- **Ressourcennutzung**: Berücksichtigen Sie Speicher- und CPU-Auslastung bei mehreren gleichzeitig aktiven Agenten
- **Fehlerbehandlung**: Implementieren Sie robuste Fehlererholung für parallele Operationen

### 🏗️ **Architektur gleichzeitiger Workflows**

```mermaid
graph TD
    A[Arbeitsablauf Start] --> B[Gleichzeitige Ausführung]
    B --> C[Agenten-Pool 1]
    B --> D[Agenten-Pool 2]
    B --> E[Agenten-Pool 3]
    C --> F[Ergebnisaggregation]
    D --> F
    E --> F
    F --> G[Endausgabe]
    
    H[Microsoft Foundry] --> C
    H --> D
    H --> E
```

**Wesentliche Vorteile:**
- **⚡ Leistung**: Deutliche Beschleunigung durch parallele Ausführung
- **📈 Skalierbarkeit**: Erhöhte Arbeitslast bewältigen, ohne proportional mehr Zeit zu benötigen
- **🔄 Effizienz**: Bessere Nutzung der verfügbaren Rechenressourcen
- **🎯 Durchsatz**: Mehr Arbeit in derselben Zeit verarbeiten

## 🎨 **Designmuster für gleichzeitige Workflows**

### 🔍 **Forschungs- & Analyse-Pipeline**
```
Research Task → Parallel Research Agents → Content Synthesis → Quality Review
```

### 📊 **Datenverarbeitungs-Workflow**
```
Input Data → Concurrent Processing Agents → Result Aggregation → Final Report
```

### 🎭 **Inhaltserstellungspipeline**
```
Content Brief → Parallel Content Generators → Review & Merge → Final Content
```

### 🔄 **Mehrstufige Verarbeitung**
```
Input → Stage 1 (Concurrent) → Stage 2 (Concurrent) → Stage 3 (Sequential) → Output
```

## 🏢 **Leistungsvorteile für Unternehmen**

### ⚡ **Durchsatzoptimierung**
- **Parallele Ausführung**: Mehrere Agenten arbeiten gleichzeitig
- **Ressourcenauslastung**: Maximale Effizienz der verfügbaren KI-Modellkapazität
- **Zeitreduktion**: Deutliche Verringerung der Gesamtverarbeitungszeit
- **Skalierbare Architektur**: Bei Bedarf weitere gleichzeitige Agenten einfach hinzufügen

### 🛡️ **Zuverlässigkeit & Widerstandsfähigkeit**
- **Fehlertoleranz**: Einzelne Agentenausfälle stoppen nicht den gesamten Workflow
- **Fehlerisolation**: Probleme in einem parallelen Zweig beeinflussen die anderen nicht
- **Sanfter Abbau**: Das System läuft auch bei verminderter Agentenkapazität weiter
- **Wiederherstellungsmechanismen**: Automatische Wiederholungen und Fehlerbehandlung fehlgeschlagener Operationen

### 📊 **Überwachung & Beobachtbarkeit**
- **Verfolgung paralleler Ausführung**: Fortschritt aller parallelen Operationen überwachen
- **Leistungskennzahlen**: Beschleunigung und Effizienzgewinne messen
- **Analyse der Ressourcennutzung**: Gleichzeitige Agentenzuweisung optimieren
- **Engpassidentifikation**: Leistungsengpässe finden und beheben

Lassen Sie uns leistungsstarke parallele KI-Workflows erstellen! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
import os
from typing import Any

from agent_framework import (
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowViz,
    handler,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
ResearcherAgentName = "Researcher-Agent"
ResearcherAgentInstructions = "You are my travel researcher, working with me to analyze the destination, list relevant attractions, and make detailed plans for each attraction."

In [ ]:
PlanAgentName = "Plan-Agent"
PlanAgentInstructions = "You are my travel planner, working with me to create a detailed travel plan based on the researcher's findings."

In [ ]:
research_agent = provider.as_agent(
    name=ResearcherAgentName,
    instructions=ResearcherAgentInstructions,
)

plan_agent = provider.as_agent(
    name=PlanAgentName,
    instructions=PlanAgentInstructions,
)


In [ ]:
# A passthrough executor that broadcasts the user input to every agent in parallel.
class InputDispatcher(Executor):
    """Forward the user input unchanged to all participating agents."""

    @handler
    async def forward(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text)


dispatcher = InputDispatcher(id="dispatcher")
agents = [research_agent, plan_agent]

workflow = (
    WorkflowBuilder(
        start_executor=dispatcher,
        output_executors=agents,
    )
    .add_fan_out_edges(dispatcher, agents)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
events = await workflow.run("Plan a trip to Seattle in December")
outputs = events.get_outputs()

In [ ]:
if outputs:
    print("===== Final Aggregated Responses =====")
    # outputs is a list of AgentResponse objects, one per output executor
    # (research_agent then plan_agent), in the order given to output_executors.
    for i, response in enumerate(outputs, start=1):
        print(f"{'-' * 60}\n\n{i:02d}:\n{response.text}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache gilt als maßgebliche Quelle. Bei kritischen Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
